- 제공된 학습용 데이터(elec_train.csv)는 전국 건물의 기상 정보(기온, 강수량, 풍속, 습도) 및 전력 소비량을 기록한 자료이다. <br>
    해당 데이터를 기반으로 전력 소비량을 예측하는 회귀 모델을 개발하고, 가장 우수한 모델을 평가 데이터(elec_test.csv)에 적용하여 <br>
    전력 소비량을 예측하시오. 예측 결과는 아래의 [제출 형식]을 준수하여, CSV 파일로 생성하는 코드를 제출하시오.

> [제출형식] CSV 파일명: result.csv, 전력 소비량 컬럼명: pred (1개)

In [52]:
# 출력을 원하실 경우 print() 함수 활용
# 예시) print(df.head())

# getcmd(), chdir() 등 작업 폴더 설정 불필요
# 파일 경로 상 내부 드라이버 경로(C: 등) 접근 불가

import pandas as pd

train = pd.read_csv('../sample_data/part2/회귀/전기사용량/elec_train.csv')
test = pd.read_csv('../sample_data/part2/회귀/전기사용량/elec_test.csv')

# 답안 제출 참고
# 아래 코드는 예시이며 변수명 등 개인별로 변경하여 활용
# pd.DataFrame변수.to_csv('result.csv', index=False)

#train.info()
#test.info()

# 건물코드 object
# 기온,강수량,풍속,습도,전력소비량 float64
# test - 전력소비량 x (종속변수)

# 결측값 찾기
#train.isnull().sum() # 강수량, 풍속, 습도

# train 데이터 종속변수/독립변수 나누기
X_train = train.drop('전력소비량', axis=1)
y = train['전력소비량']

# 결측값 채우기
X_train['강수량'] = X_train['강수량'].fillna(0)
X_train['풍속'] = X_train['풍속'].fillna(X_train['풍속'].mean())
X_train['습도'] = X_train['습도'].fillna(X_train['습도'].mode()[0])

# 수치형 데이터 스케일링
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# 수치형 데이터만 추출하기
num_columns = X_train.select_dtypes(exclude='object').columns
X_train[num_columns] = scaler.fit_transform(X_train[num_columns])
test[num_columns] = scaler.transform(test[num_columns])

# 범주형 데이터 인코딩하기
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

X_train['건물코드'] = encoder.fit_transform(X_train['건물코드'])
test['건물코드'] = encoder.transform(test['건물코드'])

# 학습/검증 데이터 나누기
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2)
# print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# 학습하기
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()
model.fit(X_train, y_train)

# 검증하기
from sklearn.metrics import mean_squared_error, r2_score
y_pred = model.predict(X_val)
mse = mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
# print(mse, r2)

# test 데이터 예측하기
test_pred = model.predict(test)
result = pd.DataFrame(test_pred, columns=['pred'])
result.to_csv('result.csv', index=False)
